# FABS Track 2 — RL Maze Agent · Colab Trainer

End-to-end notebook: clones the repo, generates maps, trains PPO, runs the
benchmark, and renders all visualizations.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Expected wall-clock on T4 (free tier):**
- maps generation: ~30 s
- train (2.5M steps): ~12 min
- benchmark (100 ep × 19 maps): ~5 min
- ablation study (optional, +8 rubric points): ~50 min

If the repo is already cloned in this Colab session, skip Section 1 and
start at Section 1b (`git pull`).

## 1. First-time setup — clone the repo

In [ ]:
!git clone https://github.com/Shalbulov/ai-hackathon-t2-maze-rl.git
%cd ai-hackathon-t2-maze-rl

## 1b. If repo already cloned — pull latest changes

In [ ]:
%cd /content/ai-hackathon-t2-maze-rl
!git pull

## 2. Install dependencies + verify GPU

Re-run this cell every time the Colab runtime reconnects (Colab wipes installed pip packages on disconnect).

In [ ]:
!pip install -q -r requirements.txt
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU not attached — Runtime → Change runtime type → T4 GPU'

## 3. Generate maps (16 train + 3 test)

All 19 maps drawn from the same procedural distribution. Test set are 3 unseen topologies — generalization is measured across layout, not across distributions.

In [ ]:
!python maze_gen.py --train 16 --test 3 --size 9 --seed 42
!ls maps/

## 4. Quick env smoke test (5 random steps)

Sanity check that env loads and observation/action spaces are correct.

In [ ]:
from env import Maze3DEnv
from env.Maze3DEnv import N_OBS
import numpy as np
env = Maze3DEnv('maps/train1.npy', randomize=True)
obs, info = env.reset(seed=0)
assert obs.shape == (N_OBS,), obs.shape
print(f'obs shape: {obs.shape}  (23 spec + 4 visit-count = {N_OBS})')
for _ in range(5):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    print(f'step={env.steps} r={r:+.3f} pos={info["pos"]}')
print('OK')

## 5. Train PPO (2.5M steps, ~12 min on T4)

Memoryless PPO + 27-dim observation (23 spec features + 4 visit-count). Vectorized raycast brings env throughput to ~10k steps/sec, so 2.5M timesteps fits in ~12 min on T4.

**Reward stack:**
- `-0.02` time penalty
- `+1.0 × bfs_progress` — BFS-distance shaping (reachable-path progress)
- `+0.1` first-visit exploration bonus
- `+0.02 × fric_bonus` — prefer high-grip terrain
- `-0.2` on wall hit
- `+50` on goal

**Watch the progress bar:**

| Steps | `success_rate` | `ep_rew_mean` |
|---:|---:|---:|
| 200k | 0.3–0.4 | 10–18 |
| 800k | 0.7–0.85 | 35–50 |
| 2.5M | **0.95+** | **55–65** |

**Expected benchmark result:** 16/16 train + 2/3 test, gen gap ~26%.

In [ ]:
!python train.py --steps 2500000 --n-envs 8 --seed 42

## 6. Benchmark (100 ep × 19 maps + visualizations)

Loads `agents/best_model.zip` by default. The benchmark writes
`viz/results_table.txt`, per-map heatmaps, before/after GIF, and 3D trajectory.

In [ ]:
!python benchmark.py --model agents/best_model.zip --episodes 100

## 6b. Display results inline

In [ ]:
from IPython.display import Image, display
print(open('viz/results_table.txt').read())
print('\n--- before/after (random vs trained) ---')
display(Image('viz/before_after.gif'))
print('\n--- visit heatmap on train1 ---')
display(Image('viz/heatmap_train1.png'))
print('\n--- 3D trajectory (z = timestep) ---')
display(Image('viz/trajectory_3d.gif'))
print('\n--- learning curve ---')
display(Image('viz/learning_curve.png'))

## 7. Ablation study (~50 min, +8 rubric points — optional but high-value)

Trains 4 short variants (full / no_surface / no_progress / no_dr) at 100k steps each, then compares train/test step counts. Skip if running out of time — main agent is already saved.

In [ ]:
!python ablation.py --steps 100000 --episodes 20
print(open('viz/ablation.txt').read())

## 8. Download artifacts

Bundles `agents/`, `viz/`, `maps/` and `README.md` into a single zip you can download from Colab.

In [ ]:
!zip -qr submission.zip agents/best_model.zip agents/agent.zip viz/ maps/ README.md 2>/dev/null
from google.colab import files
files.download('submission.zip')

## Troubleshooting

- **`ep_rew_mean` stuck near 0 after 200k steps** — policy collapsed; lower `--n-envs` to 4 or raise `ent_coef` in `train.py`.
- **`ale-py` install error** — covered by `requirements.txt`; run `!git pull` and retry the install cell.
- **Benchmark shows 250 steps / 0% success on many maps** — undertrained; re-run training with more steps.